## Rolling-cohort CV, Catboost/LightGBM Tweedie + log1p, MAPE на holdout (RMSE/MAE контрольный)

# 03 — Baseline: LightGBM под MAPE

**Конфигурация** (с учётом дрейфа из ноутбука 02 и железа 4CPU/8GB):
- Таргет: log1p(TARGET_1), обратно expm1. Сравним с TARGET_2.
- Оптимизация под MAPE (основная метрика хакатона).
- DROP MONTH_COUNT (детерминированная метка времени, leak).
- Категориальные: REGION/BRANCH/SUBJECT — native LGBM; CITY — частотное кодирование (высокая кардинальность + 201 новый город в holdout).
- Подбор гиперов на готовом val. K-fold внутри train для стабильности.
- Тест: importance weights с весами vs без → вердикт.
- Метрики: MAPE (главная) + MAE, R², MAPE по сегментам `недоторговки` (<246083 / ≥).

In [1]:
import sys
sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold

from src.read_data import read_data          # та же функция чтения
from src.drift import compute_importance_weights

In [2]:
DATA_DIR = Path().resolve().parent / "data"
print(DATA_DIR)

/Users/alexey_macos/Documents/ITjob/Pet-projects/Magnit_retailhack_2026/data


In [3]:
train = read_data(DATA_DIR / "train.csv")
val = read_data(DATA_DIR / "val.csv")
holdout = read_data(DATA_DIR / "holdout.csv")

def mape_safe(y_true, y_pred, eps=1e-8):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mask = np.abs(y_true) > eps
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

print(f"train {train.shape}, val {val.shape}, holdout {holdout.shape}")

train (15109, 56), val (2068, 56), holdout (1322, 54)


#### Features 1st iter

In [4]:
TARGET = "TARGET_1"                 # позже сравним с TARGET_2
SEGMENT_THRESHOLD = 246083          # недоторговка в двух ценовых сегментах (< 246 083 у.е. и ≥ 246 083 у.е.)

# DROP: ID, оба таргета, MONTH_COUNT (leak-метка времени из ноутбука 02)
DROP_COLS = {"ID", "TARGET_1", "TARGET_2", "MONTH_COUNT"}

cat_native = ["REGION", "BRANCH", "SUBJECT"]   # низкая/средняя кардинальность → native LGBM
cat_freq = ["CITY"]                            # высокая кардинальность → частотное кодирование

feat_cols = [c for c in train.columns if c not in DROP_COLS]

# Частотное кодирование CITY (по train; для невиданных городов → 0)
city_freq = train["CITY"].value_counts(normalize=True)
for df in [train, val, holdout]:
    df["CITY_FREQ"] = df["CITY"].map(city_freq).fillna(0)

feat_cols = [c for c in feat_cols if c != "CITY"] + ["CITY_FREQ"]

# Native categorical для LGBM
for df in [train, val, holdout]:
    for c in cat_native:
        df[c] = df[c].astype("category")

print(f"Фичей: {len(feat_cols)} | native cat: {cat_native} | CITY→CITY_FREQ")

Фичей: 52 | native cat: ['REGION', 'BRANCH', 'SUBJECT'] | CITY→CITY_FREQ


In [5]:
def train_lgbm(X_tr, y_tr, X_va, y_va, cat_feats, objective="regression",
               sample_weight=None, n_round=1500, seed=42):
    """LightGBM с log1p target. objective: 'regression'(на логах≈MAPE), 'mape', 'tweedie'."""
    params = dict(objective=objective, metric="mae", learning_rate=0.03,
                  num_leaves=31, min_data_in_leaf=40, feature_fraction=0.8,
                  bagging_fraction=0.8, bagging_freq=5, lambda_l1=0.1, lambda_l2=1.0,
                  num_threads=4, seed=seed, verbose=-1)
    if objective == "tweedie":
        params["tweedie_variance_power"] = 1.5

    ds_tr = lgb.Dataset(X_tr, np.log1p(y_tr), categorical_feature=cat_feats,
                        weight=sample_weight, free_raw_data=False)
    ds_va = lgb.Dataset(X_va, np.log1p(y_va), categorical_feature=cat_feats,
                        free_raw_data=False)
    m = lgb.train(params, ds_tr, num_boost_round=n_round, valid_sets=[ds_va],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    return m

def predict(m, X):
    return np.clip(np.expm1(m.predict(X, num_iteration=m.best_iteration)),
                   100_000, 1_000_000)   # clip к границам таргета (из EDA)

# Сравниваем 3 objective на готовом val
X_tr, y_tr = train[feat_cols], train[TARGET]
X_va, y_va = val[feat_cols], val[TARGET]

print("Objective       MAPE      MAE       R²")
for obj in ["regression", "mape", "tweedie"]:
    m = train_lgbm(X_tr, y_tr, X_va, y_va, cat_native, objective=obj)
    p = predict(m, X_va)
    print(f"{obj:14s}  {mape_safe(y_va, p):6.2f}%  {mean_absolute_error(y_va, p):8.0f}  "
          f"{r2_score(y_va, p):+.3f}  (best_iter={m.best_iteration})")

Objective       MAPE      MAE       R²


regression       19.74%     42132  +0.445  (best_iter=831)
mape             20.21%     43204  +0.415  (best_iter=828)
tweedie          19.88%     42487  +0.438  (best_iter=805)


!!!
1. `regression на log1p` выиграл — ровно как мы и предполагали: MSE на логах ≈ относительная ошибка ≈ то, что штрафует MAPE. Прямой mape-objective оказался хуже всех (численно капризен, как и ожидалось). Tweedie на втором месте, но log1p уже сделал его работу (двойная коррекция хвоста не помогла).

2. Берём objective='regression' на log1p как основу.

`MAPE ~20%` --> Это baseline-ориентир:

- Это одна модель без ансамбля, без коррекции, без тюнинга.
- На дрейфующем holdout будет, вероятно, хуже (val ближе к train, чем holdout; holdout — самые новые магазины, MONTH_COUNT 1-12).
- R² +0.445 — модель объясняет ~45% дисперсии. Адекватно для cold-start с дрейфом, не катастрофа.

`Важная оговорка — это MAPE на VAL, не holdout`

- Val имеет MONTH_COUNT 13-36, holdout 1-12. Holdout дальше от train → реальный MAPE на сдаче будет выше. 

- Это нормально, главное — относительное улучшение (ансамбль, коррекция должны снижать).

In [6]:
# вердикт по importance weights

# Веса считаем БЕЗ MONTH_COUNT (он уже не в feat_cols, но в src-функцию передаём чистый список)
feat_for_weights = [c for c in feat_cols if c != "CITY_FREQ"] + ["CITY"]  # weights на сырых
# проще: используем те же feat_cols (CITY_FREQ уже числовая)
w = compute_importance_weights(train, holdout, feat_cols, cat_text=cat_native)

best_obj = "regression"

print("Importance weights — вердикт на val:")
for label, sw in [("БЕЗ весов", None), ("С весами", w)]:
    m = train_lgbm(X_tr, y_tr, X_va, y_va, cat_native, objective=best_obj, sample_weight=sw)
    p = predict(m, X_va)
    print(f"  {label:12s}: MAPE={mape_safe(y_va, p):.2f}%  MAE={mean_absolute_error(y_va, p):.0f}")

Importance weights — вердикт на val:
  БЕЗ весов   : MAPE=19.74%  MAE=42132
  С весами    : MAPE=19.65%  MAE=42173


Подтвердилось ожидание: 

- при AUC 0.963 importance weighting структурно ограничен. Веса бимодальные (57% магазинов прижаты к минимуму), эффективная выборка схлопнута — поэтому эффект почти нулевой. MAE даже чуть хуже (42132 → 42173).

Вердикт по reweighting:

−0.09 п.п. — это шум, не сигнал. Не стоит закладывать веса в финальное решение как основной приём. 

Но:

Выигрыш в правильную сторону (MAPE чуть лучше) — значит веса не вредят.
Можно оставить опционально в ансамбле (одна из моделей с весами для разнообразия), но не делать на них ставку.

Для презентации:

«Дрейф настолько сильный (adversarial AUC 0.963), что importance weighting через density-ratio схлопывается: основную ставку делаем не на reweighting, а на `drift-robust` признаки и `monotone constraints` (причинные связи устойчивее к дрейфу, чем веса)».

In [7]:
# MAPE по сегментам + сравнение TARGET_1/TARGET_2

m = train_lgbm(X_tr, y_tr, X_va, y_va, cat_native, objective=best_obj)
p = predict(m, X_va)

print("MAPE по сегментам (TARGET_1):")
for name, mask in [("low <246083", y_va < SEGMENT_THRESHOLD),
                   ("high ≥246083", y_va >= SEGMENT_THRESHOLD)]:
    print(f"  {name}: MAPE={mape_safe(y_va[mask], p[mask]):.2f}% (n={mask.sum()})")

# Недоторговка: факт < 90% прогноза
under = (y_va < 0.9 * p).mean() * 100
print(f"Доля недоторговок (факт<90% прогноза): {under:.1f}%")

# TARGET_2 для сравнения
m2 = train_lgbm(train[feat_cols], train["TARGET_2"], val[feat_cols], val["TARGET_2"],
                cat_native, objective=best_obj)
p2 = predict(m2, val[feat_cols])
print(f"\nTARGET_1 MAPE: {mape_safe(y_va, p):.2f}%")
print(f"TARGET_2 MAPE: {mape_safe(val['TARGET_2'], p2):.2f}%")

MAPE по сегментам (TARGET_1):
  low <246083: MAPE=22.99% (n=1396)
  high ≥246083: MAPE=12.99% (n=672)
Доля недоторговок (факт<90% прогноза): 50.8%

TARGET_1 MAPE: 19.74%
TARGET_2 MAPE: 19.10%


In [8]:
# Быстрая проверка стабильности таргета

def cv_mape(target_col, n_splits=5, seed=42):
    scores = []
    kf = KFold(n_splits, shuffle=True, random_state=seed)
    for tr_idx, va_idx in kf.split(train):
        tr_f, va_f = train.iloc[tr_idx], train.iloc[va_idx]
        m = train_lgbm(tr_f[feat_cols], tr_f[target_col],
                       va_f[feat_cols], va_f[target_col], cat_native, objective=best_obj)
        p = predict(m, va_f[feat_cols])
        scores.append(mape_safe(va_f[target_col], p))
    return np.mean(scores), np.std(scores)

for tgt in ["TARGET_1", "TARGET_2"]:
    mean, std = cv_mape(tgt)
    print(f"{tgt}: CV MAPE = {mean:.2f}% ± {std:.2f}%")

TARGET_1: CV MAPE = 14.07% ± 0.17%
TARGET_2: CV MAPE = 14.12% ± 0.14%


!!!

Сравнение CV MAPE TARGET_1/TARGET_2:

Разрыв 14% vs 20% — это не шум, это сигнал дрейфа. K-fold на train даёт ~14% (train-магазины предсказываем train-магазинами, распределения совпадают). На val — ~20% (val-магазины дрейфуют относительно train). Разница ~6 п.п. — это цена дрейфа в чистом виде.


In [9]:
# Если T2 как основной, проверка влияния MONTH_COUNT

for use_month in [False, True]:
    fc = feat_cols if not use_month else feat_cols + ["MONTH_COUNT"]
    # (для use_month=True добавь MONTH_COUNT обратно в train/val/holdout)
    m = train_lgbm(train[fc], train["TARGET_2"], val[fc], val["TARGET_2"],
                   cat_native, objective="regression")
    p = predict(m, val[fc])
    print(f"MONTH_COUNT={use_month}: T2 MAPE на val = {mape_safe(val['TARGET_2'], p):.2f}%")

MONTH_COUNT=False: T2 MAPE на val = 19.10%
MONTH_COUNT=True: T2 MAPE на val = 17.76%


In [10]:
print(f"corr(MONTH_COUNT, T2): {train['MONTH_COUNT'].corr(train['TARGET_2']):.3f}")
print(f"corr(MONTH_COUNT, T1): {train['MONTH_COUNT'].corr(train['TARGET_1']):.3f}")

corr(MONTH_COUNT, T2): -0.065
corr(MONTH_COUNT, T1): -0.056


In [11]:
# проверить безопасность MONTH_COUNT для MAPE T1/T2 - может зря дропнули

# Эмуляция: train на старых (MONTH_COUNT >= 60), val на молодых train (37-59)
# Это имитирует экстраполяцию к меньшему возрасту, как val/holdout
old = train[train["MONTH_COUNT"] >= 60]
young = train[train["MONTH_COUNT"] < 60]

for use_month in [False, True]:
    fc = feat_cols + (["MONTH_COUNT"] if use_month else [])
    m = train_lgbm(old[fc], old["TARGET_2"], young[fc], young["TARGET_2"],
                   cat_native, objective="regression")
    p = predict(m, young[fc])
    print(f"MONTH_COUNT={use_month}: MAPE на молодых train = {mape_safe(young['TARGET_2'], p):.2f}%")

MONTH_COUNT=False: MAPE на молодых train = 15.46%
MONTH_COUNT=True: MAPE на молодых train = 15.63%


Вывод: 

- улучшение на val (−1.34 п.п.) — артефакт. На val модель «угадывает» через MONTH_COUNT, потому что val (13-36) случайно ложится на продолжение train-тренда. Но когда мы честно имитируем экстраполяцию (old→young), польза исчезает и переходит в минус.

- MONTH_COUNT не несёт переносимого сигнала — он несёт val-специфичный артефакт. Наш изначальный дроп был прав.

НО:
- дропнем только старые магазины по MONTH_COUNT
- сегментируем магазины по площади TRADE_SQUARE
- обучаем по `TARGET_2`

#### Feature engineering

In [12]:
# ============ Подготовка: FE + drift-robust признаки ============
def add_features(df):
    """Переносимые FE-фичи (без MONTH_COUNT-производных)."""
    df = df.copy()
    eps = 1e-6
    
    df["FAMILIES_PER_SQ"]  = df["HUFFFAMILIES"] / (df["TRADE_SQUARE"] + eps)
    df["RELATIVE_FAM_SQ"]  = df["HUFFRELATIVEFAMILIES"] / (df["TRADE_SQUARE"] + eps)
    df["RENT_DENSITY"]     = df["RENT_HEX"] / (df["TRADE_SQUARE"] + eps)
    df["COMP_PRESSURE"]    = df["HUFF_RANK_COMPS_CANNIBALS"] + df["TOTAL_RANK_COMPS_CANNIBALS"]
    df["INFRA_SCORE"]      = (df["HOSPITAL"] + df["KINDERGARTENS"] + df["BANKS"]
                              + df["COLLEGES"] + df["METRO"] + df["STATIONS"])
    df["TRANSPORT_ACCESS"] = (df["ROUTES_CNT"] + df["CROSSWALK"] + df["CROSSROAD"]
                              + df["HUB"] + df["ON_MAIN_CITY_ROAD"])
    df["HAS_LICENSE"]      = df["ALCOHOL"] + df["TOBACCO"]
    df["SQUARE_LOG"]         = np.log1p(df["TRADE_SQUARE"])
    df["DIST_TO_CENTER_LOG"] = np.log1p(df["DIST_TO_ADM_CENTER"])
    return df

def add_drift_robust(df, ref_train):
    """Drift-robust: аренда относительно медианы региона. Статистики из ref_train (против leak)."""
    df = df.copy()
    eps = 1e-6
    # ref_train["REGION"] тоже может быть category — берём как str для groupby
    rent_med = ref_train.groupby(ref_train["REGION"].astype(str), observed=True)["RENT_HEX"].median()
    global_med = ref_train["RENT_HEX"].median()
    mapped = df["REGION"].astype(str).map(rent_med)        # str → числовой результат map
    mapped = mapped.fillna(global_med).clip(lower=eps)      # теперь fillna работает (float, не category)
    df["RENT_REL_REGION"] = df["RENT_HEX"] / mapped
    return df

# Применяем последовательно: сырьё → FE → drift-robust
train_x = add_drift_robust(add_features(train), add_features(train))
val_x   = add_drift_robust(add_features(val),   add_features(train))
holdout_x = add_drift_robust(add_features(holdout), add_features(train))

# Списки фичей
new_fe = ["FAMILIES_PER_SQ", "RELATIVE_FAM_SQ", "RENT_DENSITY", "COMP_PRESSURE",
          "INFRA_SCORE", "TRANSPORT_ACCESS", "HAS_LICENSE", "SQUARE_LOG", "DIST_TO_CENTER_LOG"]
feat_fe = feat_cols + new_fe                        # базовые + FE
feat_dr = feat_fe + ["RENT_REL_REGION"]             # + drift-robust

# Функция двойного замера на TARGET_2
TGT = "TARGET_2"
old_x   = train_x[train_x["MONTH_COUNT"] >= 60]
young_x = train_x[train_x["MONTH_COUNT"] < 60]

def eval_both(fc, train_df=train_x):
    """Возвращает (val_mape, old→young_mape) на TARGET_2."""
    m1 = train_lgbm(train_df[fc], train_df[TGT], val_x[fc], val_x[TGT], cat_native, objective="regression")
    val_mape = mape_safe(val_x[TGT], predict(m1, val_x[fc]))
    o = train_df[train_df["MONTH_COUNT"] >= 60]
    y = train_df[train_df["MONTH_COUNT"] < 60]
    m2 = train_lgbm(o[fc], o[TGT], y[fc], y[TGT], cat_native, objective="regression")
    oy_mape = mape_safe(y[TGT], predict(m2, y[fc]))
    return val_mape, oy_mape

print("Подготовка готова.")
print(f"feat_cols (база): {len(feat_cols)} | feat_fe: {len(feat_fe)} | feat_dr: {len(feat_dr)}")

Подготовка готова.
feat_cols (база): 52 | feat_fe: 61 | feat_dr: 62


In [13]:
print("Набор фичей                  val      old→young")
v, o = eval_both(feat_cols); print(f"База (без FE):              {v:.2f}%   {o:.2f}%")
v, o = eval_both(feat_fe);   print(f"+ FE-фичи:                  {v:.2f}%   {o:.2f}%")
v, o = eval_both(feat_dr);   print(f"+ drift-robust (RENT_REL):  {v:.2f}%   {o:.2f}%")

Набор фичей                  val      old→young
База (без FE):              19.10%   15.46%
+ FE-фичи:                  19.11%   15.53%
+ drift-robust (RENT_REL):  19.06%   15.54%


Все три практически идентичны. Разброс ±0.05 п.п. — это шум, не сигнал.

Вывод — FE и drift-robust НЕ помогли

- FE-фичи: val 19.10→19.11 (хуже на 0.01), old→young 15.46→15.53 (хуже на 0.07). Не улучшили ничего.
- Drift-robust: val 19.06 (лучше на 0.04), old→young 15.54 (хуже на 0.08). Микроскопический выигрыш на val, проигрыш на экстраполяции = шум.

1. Берём базовый набор `feat_cols` (52 фичи). FE и drift-robust не дают переносимого прироста — добавлять 9-10 фичей ради нуля не стоит.

2. Исключение: можно оставить RENT_REL_REGION (единственная, что чуть улучшила val), но честно — это шум. Рекомендую чистый feat_cols.

In [14]:
feat_best = feat_cols   # базовый набор — FE не помог

print("Train filter            val MAPE T2   n_train")
for thr in [None, 150, 100, 80, 60]:
    tr = train_x if thr is None else train_x[train_x["MONTH_COUNT"] < thr]
    if len(tr) < 2000:
        print(f"  < {str(thr):5s}: пропуск (n_train={len(tr)})")
        continue
    m = train_lgbm(tr[feat_best], tr[TGT], val_x[feat_best], val_x[TGT],
                   cat_native, objective="regression")
    p = predict(m, val_x[feat_best])
    print(f"  < {str(thr):5s}              {mape_safe(val_x[TGT], p):.2f}%      {len(tr)}")

Train filter            val MAPE T2   n_train
  < None               19.10%      15109
  < 150                18.92%      10251
  < 100                18.87%      6172
  < 80                 18.91%      3599
  < 60                 19.06%      2126


In [15]:
tr100 = train_x[train_x["MONTH_COUNT"] < 100]
o = tr100[tr100["MONTH_COUNT"] >= 60]
y = tr100[tr100["MONTH_COUNT"] < 60]
m = train_lgbm(o[feat_cols], o[TGT], y[feat_cols], y[TGT], cat_native, objective="regression")
print(f"old→young на фильтре <100: {mape_safe(y[TGT], predict(m, y[feat_cols])):.2f}%")

old→young на фильтре <100: 15.75%


Не использовать фильтр < 100 — придерживаться всех данных (old→young честнее, а он за полный train).

Использовать мягкий фильтр <150 — он давал val 18.92 (почти как <100), но оставляет 10251 строк (больше данных → меньше риск переэкстраполяции). Компромисс.

In [16]:
sq_median = train["TRADE_SQUARE"].median()
for df in [train_x, val_x, holdout_x]:
    df["IS_LARGE_STORE"] = (df["TRADE_SQUARE"] >= sq_median).astype(int)

feat_seg = feat_cols + ["IS_LARGE_STORE"]

print("Конфигурация              val      old→young")
v, o = eval_both(feat_cols); print(f"без сегмента:            {v:.2f}%   {o:.2f}%")
v, o = eval_both(feat_seg);  print(f"+ IS_LARGE_STORE:        {v:.2f}%   {o:.2f}%")

Конфигурация              val      old→young
без сегмента:            19.10%   15.46%
+ IS_LARGE_STORE:        19.04%   15.49%


- val чуть лучше (−0.06), old→young чуть хуже (+0.03). 
- Снова в пределах шума (±0.06 п.п.), разнонаправленно. 
- LightGBM уже ловит «большой/малый» через сырой TRADE_SQUARE, бинарный флаг ничего не добавляет.

In [17]:
# Продвинутое Раздельные модели по площади 

sq_median = train["TRADE_SQUARE"].median()
print(f"Медиана TRADE_SQUARE: {sq_median:.4f}")

def eval_separate_by_size(train_df, val_df, fc):
    """Две модели: на малых и крупных магазинах отдельно. Возвращает общий MAPE на val."""
    preds = np.zeros(len(val_df))
    for is_large in [0, 1]:
        # train-срез данного сегмента
        tr_seg = train_df[(train_df["TRADE_SQUARE"] >= sq_median) == bool(is_large)]
        # val-маска данного сегмента
        va_mask = ((val_df["TRADE_SQUARE"] >= sq_median) == bool(is_large)).to_numpy()
        if va_mask.sum() == 0 or len(tr_seg) < 500:
            continue
        m = train_lgbm(tr_seg[fc], tr_seg[TGT],
                       val_df[va_mask][fc], val_df[va_mask][TGT],
                       cat_native, objective="regression")
        preds[va_mask] = predict(m, val_df[va_mask][fc])
        print(f"  сегмент {'крупные' if is_large else 'малые':8s}: "
              f"n_train={len(tr_seg)}, n_val={va_mask.sum()}, "
              f"MAPE={mape_safe(val_df[va_mask][TGT], preds[va_mask]):.2f}%")
    return mape_safe(val_df[TGT], preds)

# Единая модель (референс)
m_single = train_lgbm(train_x[feat_cols], train_x[TGT], val_x[feat_cols], val_x[TGT],
                      cat_native, objective="regression")
single_mape = mape_safe(val_x[TGT], predict(m_single, val_x[feat_cols]))

print("Раздельные модели по площади:")
sep_mape = eval_separate_by_size(train_x, val_x, feat_cols)

print(f"\n  Единая модель: MAPE = {single_mape:.2f}%")
print(f"  Раздельные модели:  MAPE = {sep_mape:.2f}%")

Медиана TRADE_SQUARE: 0.6097
Раздельные модели по площади:
  сегмент малые   : n_train=7554, n_val=886, MAPE=18.63%
  сегмент крупные : n_train=7555, n_val=1182, MAPE=19.82%

  Единая модель: MAPE = 19.10%
  Раздельные модели:  MAPE = 19.31%


Вывод

1. Раздельные модели проиграли единой. Причина — дробление данных: каждая модель учится на ~7.5k вместо 15k. При дрейфе меньше данных = хуже обобщение. Специализация под формат не окупила потерю объёма. Тот же урок, что с фильтром старых: при дрейфе данные ценнее тонкой настройки.

2. Интересная деталь: малые магазины по площади (18.63%) предсказываются лучше крупных (19.82%). Это противоположно ценовому сегменту (там малые по продажам были хуже — 23%). Подтверждает: площадь ≠ продажи — разные оси. Деление по площади не бьёт в наш слабый ценовой сегмент.

Берем:

Фичи (feat_cols):     52 колонки, БЕЗ MONTH_COUNT

Строки (train):       все 15109, БЕЗ фильтра по возрасту магазина

Таргет:               TARGET_2

MONTH_COUNT:          остаётся в датафрейме как служебный столбец
                      (для old→young теста и диагностики), но НЕ фича

#### Encemble IDEA: LightGBM + CatBoost + RidgeCV [LGBM, CatBoost, Ridge]

In [18]:
from catboost import CatBoostRegressor

In [ ]:
# CatBoost сам ест категориальные строки — отдаём ему СЫРЫЕ REGION/BRANCH/SUBJECT/CITY
cat_cols_cb = ["REGION", "BRANCH", "SUBJECT", "CITY"]

# Фичи для CatBoost: те же базовые, но CITY вместо CITY_FREQ (CatBoost сам закодирует)
feat_cb = [c for c in feat_cols if c != "CITY_FREQ"] + ["CITY"]

def train_catboost(X_tr, y_tr, X_va, y_va, cat_features, seed=42):
    """CatBoost на log1p target. cat_features — список имён категориальных колонок."""
    # CatBoost требует категориальные как str без NaN
    X_tr, X_va = X_tr.copy(), X_va.copy()
    for c in cat_features:
        X_tr[c] = X_tr[c].astype(str)
        X_va[c] = X_va[c].astype(str)
    m = CatBoostRegressor(
        iterations=3000, learning_rate=0.03, depth=6,
        l2_leaf_reg=3.0, loss_function="RMSE",
        random_seed=seed, thread_count=4,           # 4 CPU как в Docker
        early_stopping_rounds=100, verbose=0,
    )
    m.fit(X_tr, np.log1p(y_tr), cat_features=cat_features,
          eval_set=(X_va, np.log1p(y_va)), use_best_model=True)
    return m

def predict_cb(m, X, cat_features):
    X = X.copy()
    for c in cat_features:
        X[c] = X[c].astype(str)
    return np.clip(np.expm1(m.predict(X)), 100_000, 1_000_000)

# --- Обучение и замер на val (TARGET_2) ---
m_cb = train_catboost(train_x[feat_cb], train_x[TGT], val_x[feat_cb], val_x[TGT], cat_cols_cb)
p_cb = predict_cb(m_cb, val_x[feat_cb], cat_cols_cb)
cb_val = mape_safe(val_x[TGT], p_cb)

# old→young для проверки переносимости
o = train_x[train_x["MONTH_COUNT"] >= 60]
y = train_x[train_x["MONTH_COUNT"] < 60]
m_cb_oy = train_catboost(o[feat_cb], o[TGT], y[feat_cb], y[TGT], cat_cols_cb)
cb_oy = mape_safe(y[TGT], predict_cb(m_cb_oy, y[feat_cb], cat_cols_cb))

print(f"CatBoost:  val={cb_val:.2f}%   old→young={cb_oy:.2f}%")
print("LightGBM:  val=19.10%   old→young=15.46%   (референс)")

CatBoost:  val=18.98%   old→young=15.30%
LightGBM:  val=19.10%   old→young=15.46%   (референс)


In [20]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

In [ ]:
# Ridge не ест категории и чувствителен к масштабу.
# Кодируем REGION/BRANCH/SUBJECT частотно (как CITY_FREQ), статистики из train (против leak).
def add_freq_encoding(df, ref_train, cols):
    df = df.copy()
    for c in cols:
        freq = ref_train[c].astype(str).value_counts(normalize=True)
        df[c + "_FREQ"] = df[c].astype(str).map(freq).fillna(0)
    return df

freq_cols = ["REGION", "BRANCH", "SUBJECT"]
train_r = add_freq_encoding(train_x, train_x, freq_cols)
val_r   = add_freq_encoding(val_x, train_x, freq_cols)

# Числовые фичи + freq-кодировки, без сырых категорий
ridge_feats = ([c for c in feat_cols if c not in ["REGION", "BRANCH", "SUBJECT", "CITY"]]
               + [c + "_FREQ" for c in freq_cols])
ridge_feats = [c for c in ridge_feats if c in train_r.columns]

def train_ridge(X_tr, y_tr, X_va, y_va):
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(X_tr.fillna(0))
    Xva_s = scaler.transform(X_va.fillna(0))
    m = RidgeCV(alphas=np.logspace(-2, 4, 20))
    m.fit(Xtr_s, np.log1p(y_tr))
    pred = np.clip(np.expm1(m.predict(Xva_s)), 100_000, 1_000_000)
    return m, scaler, pred

m_r, scaler_r, p_ridge = train_ridge(train_r[ridge_feats], train_r[TGT],
                                     val_r[ridge_feats], val_r[TGT])
ridge_val = mape_safe(val_r[TGT], p_ridge)

print(f"RidgeCV:   val={ridge_val:.2f}%")
print("CatBoost:  val=18.98%  | LightGBM: val=19.10%  (референс)")
print(f"Лучший alpha: {m_r.alpha_:.3f}")

RidgeCV:   val=22.27%
CatBoost:  val=18.98%  | LightGBM: val=19.10%  (референс)
Лучший alpha: 127.427


In [ ]:
# --- Собираем предсказания всех трёх моделей на ОДНОМ И ТОМ ЖЕ val ---
# (модели уже обучены: m_lgb? — переобучим LGBM для чистоты, m_cb, m_r готовы)

# 1. LightGBM (на feat_cols, CITY_FREQ)
m_lgb = train_lgbm(train_x[feat_cols], train_x[TGT], val_x[feat_cols], val_x[TGT],
                   cat_native, objective="regression")
p_lgb = predict(m_lgb, val_x[feat_cols])

# 2. CatBoost (на feat_cb, сырой CITY)
p_cb = predict_cb(m_cb, val_x[feat_cb], cat_cols_cb)

# 3. Ridge (на ridge_feats, freq-кодировки + scaler)
val_r_s = scaler_r.transform(val_r[ridge_feats].fillna(0))
p_ridge = np.clip(np.expm1(m_r.predict(val_r_s)), 100_000, 1_000_000)

y_true = val_x[TGT].to_numpy()

# Контроль: все на одной длине и в одном порядке
assert len(p_lgb) == len(p_cb) == len(p_ridge) == len(y_true)

print("Одиночные модели на val:")
print(f"  LightGBM: {mape_safe(y_true, p_lgb):.2f}%")
print(f"  CatBoost: {mape_safe(y_true, p_cb):.2f}%")
print(f"  Ridge:    {mape_safe(y_true, p_ridge):.2f}%")

# --- Способ 1: простое среднее трёх ---
p_mean3 = (p_lgb + p_cb + p_ridge) / 3
# --- Способ 1b: среднее только двух деревьев ---
p_mean2 = (p_lgb + p_cb) / 2

# --- Способ 2: взвешенное среднее (перебор весов на val) ---
best = (None, 1e9)
for wl in np.arange(0, 1.01, 0.1):
    for wc in np.arange(0, 1.01 - wl, 0.1):
        wr = 1 - wl - wc
        if wr < 0: continue
        p = wl*p_lgb + wc*p_cb + wr*p_ridge
        mp = mape_safe(y_true, p)
        if mp < best[1]:
            best = ((round(wl,1), round(wc,1), round(wr,1)), mp)

print("\nАнсамбли на val:")
print(f"  Среднее 3 (LGBM+CB+Ridge): {mape_safe(y_true, p_mean3):.2f}%")
print(f"  Среднее 2 (LGBM+CB):       {mape_safe(y_true, p_mean2):.2f}%")
print(f"  Взвеш. (w_lgb,w_cb,w_ridge)={best[0]}: {best[1]:.2f}%")
print("\n  Лучшая одиночная (CatBoost): 18.98%  ← референс")

Одиночные модели на val:
  LightGBM: 19.10%
  CatBoost: 18.98%
  Ridge:    22.27%

Ансамбли на val:
  Среднее 3 (LGBM+CB+Ridge): 19.70%
  Среднее 2 (LGBM+CB):       18.94%
  Взвеш. (w_lgb,w_cb,w_ridge)=(np.float64(0.5), np.float64(0.5), np.float64(0.0)): 18.94%

  Лучшая одиночная (CatBoost): 18.98%  ← референс


In [23]:
# external multiplier
# Берём лучшую модель (CatBoost) и подбираем множитель k на val

p_cb_val = predict_cb(m_cb, val_x[feat_cb], cat_cols_cb)
y_true = val_x[TGT].to_numpy()

print("k      val MAPE   недоторговка%")

best_k = (1.0, mape_safe(y_true, p_cb_val))
for k in np.arange(0.80, 1.06, 0.02):
    p_k = np.clip(p_cb_val * k, 100_000, 1_000_000)
    mp = mape_safe(y_true, p_k)
    under = (y_true < 0.9 * p_k).mean() * 100
    flag = " ←" if mp < best_k[1] else ""
    if mp < best_k[1]:
        best_k = (round(k,2), mp)
    print(f"{k:.2f}   {mp:.2f}%     {under:.1f}%{flag}")

print(f"\nЛучший k={best_k[0]}: MAPE={best_k[1]:.2f}% (было {mape_safe(y_true, p_cb_val):.2f}% при k=1)")

# old→young с тем же k
o = train_x[train_x["MONTH_COUNT"] >= 60]
y = train_x[train_x["MONTH_COUNT"] < 60]
m_oy = train_catboost(o[feat_cb], o[TGT], y[feat_cb], y[TGT], cat_cols_cb)
p_oy = predict_cb(m_oy, y[feat_cb], cat_cols_cb)
yt = y[TGT].to_numpy()

print("\nПроверка k на old→young:")
for k in [1.0, best_k[0]]:
    print(f"  k={k}: MAPE={mape_safe(yt, np.clip(p_oy*k, 100_000, 1_000_000)):.2f}%")

k      val MAPE   недоторговка%
0.80   16.61%     11.5% ←
0.82   15.86%     13.7% ←
0.84   15.34%     16.6% ←
0.86   15.07%     19.7% ←
0.88   15.01%     23.2% ←
0.90   15.14%     26.6%
0.92   15.51%     30.9%
0.94   16.07%     35.8%
0.96   16.83%     39.8%
0.98   17.82%     43.6%
1.00   18.98%     48.5%
1.02   20.26%     52.4%
1.04   21.66%     56.7%

Лучший k=0.88: MAPE=15.01% (было 18.98% при k=1)

Проверка k на old→young:
  k=1.0: MAPE=15.30%
  k=0.88: MAPE=15.60%


In [24]:
# сегментная коррекция по CITY_TYPE

# CatBoost прогноз на val (база для коррекции)
p_cb_val = predict_cb(m_cb, val_x[feat_cb], cat_cols_cb)
y_true = val_x[TGT].to_numpy()
city_type_val = val_x["CITY_TYPE"].to_numpy()

# Глобальный k=0.88 (референс)
global_mape = mape_safe(y_true, np.clip(p_cb_val * 0.88, 100_000, 1_000_000))
print(f"Глобальный k=0.88: val MAPE = {global_mape:.2f}%\n")

# Сегментная коррекция по CITY_TYPE: свой k для каждого типа города
print("Подбор k по сегментам CITY_TYPE:")
p_seg = p_cb_val.copy()
seg_ks = {}
for ct in sorted(np.unique(city_type_val)):
    mask = city_type_val == ct
    if mask.sum() < 20:   # маленький сегмент — глобальный k
        seg_ks[ct] = 0.88
        continue
    best = (0.88, 1e9)
    for k in np.arange(0.80, 1.02, 0.01):
        mp = mape_safe(y_true[mask], np.clip(p_cb_val[mask] * k, 100_000, 1_000_000))
        if mp < best[1]:
            best = (round(k, 2), mp)
    seg_ks[ct] = best[0]
    print(f"  CITY_TYPE={ct}: n={mask.sum():4d}, лучший k={best[0]}, MAPE={best[1]:.2f}%")

# Применяем сегментные k
for ct, k in seg_ks.items():
    p_seg[city_type_val == ct] *= k
p_seg = np.clip(p_seg, 100_000, 1_000_000)

seg_mape = mape_safe(y_true, p_seg)
print(f"\nГлобальный k=0.88:  {global_mape:.2f}%")
print(f"Сегментный k:       {seg_mape:.2f}%")

Глобальный k=0.88: val MAPE = 15.01%

Подбор k по сегментам CITY_TYPE:
  CITY_TYPE=1: n= 905, лучший k=0.88, MAPE=14.47%
  CITY_TYPE=2: n= 632, лучший k=0.89, MAPE=14.98%
  CITY_TYPE=3: n= 531, лучший k=0.85, MAPE=15.78%

Глобальный k=0.88:  15.01%
Сегментный k:       14.96%


In [25]:
# Коррекция по бинам предсказания

p_cb_val = predict_cb(m_cb, val_x[feat_cb], cat_cols_cb)
y_true = val_x[TGT].to_numpy()

# 5 бинов по предсказанию (границы из train-прогноза, чтобы применимо к holdout)
p_cb_train = predict_cb(m_cb, train_x[feat_cb], cat_cols_cb)
bin_edges = np.quantile(p_cb_train, np.linspace(0, 1, 6))  # 5 бинов
bin_edges[0], bin_edges[-1] = -np.inf, np.inf

def calibrate_by_bins(pred, y, edges, fit=True, ks=None):
    out = pred.copy()
    bins = np.digitize(pred, edges[1:-1])
    if fit:
        ks = {}
        for b in range(len(edges) - 1):
            mask = bins == b
            if mask.sum() < 20:
                ks[b] = 0.88
                continue
            best = (0.88, 1e9)
            for k in np.arange(0.78, 1.04, 0.01):
                mp = mape_safe(y[mask], np.clip(pred[mask]*k, 100000, 1000000))
                if mp < best[1]: best = (round(k,2), mp)
            ks[b] = best[0]
    for b, k in ks.items():
        out[bins == b] *= k
    return np.clip(out, 100000, 1000000), ks

# Подбор на val
p_cal, seg_ks = calibrate_by_bins(p_cb_val, y_true, bin_edges, fit=True)
print("k по бинам предсказания:")
for b, k in seg_ks.items():
    print(f"  бин {b}: k={k}")
print(f"\nГлобальный k=0.88:        {mape_safe(y_true, np.clip(p_cb_val*0.88,100000,1000000)):.2f}%")
print(f"Калибровка по 5 бинам:    {mape_safe(y_true, p_cal):.2f}%")

# Проверка переноса на old→young
o = train_x[train_x["MONTH_COUNT"]>=60]; yo = train_x[train_x["MONTH_COUNT"]<60]
m_oy = train_catboost(o[feat_cb], o[TGT], yo[feat_cb], yo[TGT], cat_cols_cb)
p_oy = predict_cb(m_oy, yo[feat_cb], cat_cols_cb)
p_oy_cal, _ = calibrate_by_bins(p_oy, yo[TGT].to_numpy(), bin_edges, fit=False, ks=seg_ks)
print(f"\nold→young глоб k=0.88:    {mape_safe(yo[TGT], np.clip(p_oy*0.88,100000,1000000)):.2f}%")
print(f"old→young по бинам:       {mape_safe(yo[TGT], p_oy_cal):.2f}%")

k по бинам предсказания:
  бин 0: k=0.89
  бин 1: k=0.87
  бин 2: k=0.87
  бин 3: k=0.87
  бин 4: k=0.89

Глобальный k=0.88:        15.01%
Калибровка по 5 бинам:    14.99%

old→young глоб k=0.88:    15.60%
old→young по бинам:       15.73%


!!! Выигрыш на val 0.02 п.п. = шум. На old→young даже хуже. Не берём.

In [ ]:
# Проверка Target encoding по географии - проверка для блендинга LightGBM + CatBoost

def add_target_encoding(train_df, other_dfs, cols, target, n_splits=5, seed=42):
    """
    TE с защитой от leak:
    - train: out-of-fold средние (kfold)
    - other (val/holdout): глобальные средние из train
    Возвращает (train_df, [other_dfs]) с новыми колонками <col>_TE.
    """
    train_df = train_df.copy()
    other_dfs = [df.copy() for df in other_dfs]
    global_mean = train_df[target].mean()

    for col in cols:
        te_col = col + "_TE"
        oof = np.zeros(len(train_df))
        kf = KFold(n_splits, shuffle=True, random_state=seed)
        for tr_idx, va_idx in kf.split(train_df):
            means = train_df.iloc[tr_idx].groupby(col, observed=True)[target].mean()
            mapped = train_df.iloc[va_idx][col].map(means).astype(float)   # ← astype(float)
            oof[va_idx] = mapped.fillna(global_mean).to_numpy()
        train_df[te_col] = oof
        full_means = train_df.groupby(col, observed=True)[target].mean()
        for df in other_dfs:
            df[te_col] = df[col].map(full_means).astype(float).fillna(global_mean)   # ← astype(float)
    return train_df, other_dfs

# TE по географии на TARGET_2
te_cols = ["REGION", "BRANCH", "CITY", "SUBJECT"]
train_te, (val_te,) = add_target_encoding(train_x, [val_x], te_cols, TGT)

# Фичи LGBM + TE (TE как числовые, плюс оставляем native categorical)
feat_lgb_te = feat_cols + [c + "_TE" for c in te_cols]

# 1) LightGBM + TE одиночка
m_lt = train_lgbm(train_te[feat_lgb_te], train_te[TGT], val_te[feat_lgb_te], val_te[TGT],
                  cat_native, objective="regression")
p_lt = predict(m_lt, val_te[feat_lgb_te])
y_true = val_x[TGT].to_numpy()

# 2) блендинг (LGBM+TE) + CatBoost
p_cb_val = predict_cb(m_cb, val_x[feat_cb], cat_cols_cb)
p_blend = 0.5 * p_lt + 0.5 * p_cb_val

print("Без коррекции (k=1):")
print("  LightGBM база:       19.10%")
print(f"  LightGBM + TE:       {mape_safe(y_true, p_lt):.2f}%")
print("  CatBoost:            18.98%")
print(f"  Блендинг (LGBT+TE,CB):{mape_safe(y_true, p_blend):.2f}%")

print("\nС коррекцией k=0.88:")
print(f"  CatBoost (текущий):  {mape_safe(y_true, np.clip(p_cb_val*0.88, 100_000, 1_000_000)):.2f}%")
print(f"  Блендинг:            {mape_safe(y_true, np.clip(p_blend*0.88, 100_000, 1_000_000)):.2f}%")

Без коррекции (k=1):
  LightGBM база:       19.10%
  LightGBM + TE:       19.08%
  CatBoost:            18.98%
  Блендинг (LGBT+TE,CB):18.95%

С коррекцией k=0.88:
  CatBoost (текущий):  15.01%
  Блендинг:            14.95%


In [27]:
# old→young: TE заново на old, применить к young (без leak)
o = train_x[train_x["MONTH_COUNT"] >= 60].copy()
yo = train_x[train_x["MONTH_COUNT"] < 60].copy()
o_te, (yo_te,) = add_target_encoding(o, [yo], te_cols, TGT)

m_lt_oy = train_lgbm(o_te[feat_lgb_te], o_te[TGT], yo_te[feat_lgb_te], yo_te[TGT],
                     cat_native, objective="regression")
p_lt_oy = predict(m_lt_oy, yo_te[feat_lgb_te])
m_cb_oy = train_catboost(o[feat_cb], o[TGT], yo[feat_cb], yo[TGT], cat_cols_cb)
p_cb_oy = predict_cb(m_cb_oy, yo[feat_cb], cat_cols_cb)
p_blend_oy = 0.5*p_lt_oy + 0.5*p_cb_oy
yt = yo[TGT].to_numpy()

print(f"old→young CatBoost k=0.88:  {mape_safe(yt, np.clip(p_cb_oy*0.88, 100_000, 1_000_000)):.2f}%")
print(f"old→young блендинг k=0.88:  {mape_safe(yt, np.clip(p_blend_oy*0.88, 100_000, 1_000_000)):.2f}%")

old→young CatBoost k=0.88:  15.60%
old→young блендинг k=0.88:  15.53%


#### Блендинг. LightGBM + Catboost с глобальным k=0.88.

In [30]:
# Подбор гиперпараметров по Optuna только для Catboost 

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [31]:
# Базовый baseline для сравнения (дефолтный CatBoost + k=0.88)
p_cb_base = predict_cb(m_cb, val_x[feat_cb], cat_cols_cb)
base_mape = mape_safe(val_x[TGT].to_numpy(), np.clip(p_cb_base * 0.88, 100_000, 1_000_000))
print(f"Дефолтный CatBoost + k=0.88: {base_mape:.2f}%\n")

y_va = val_x[TGT].to_numpy()

def objective(trial):
    params = dict(
        iterations=3000,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        depth=trial.suggest_int("depth", 4, 8),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        random_strength=trial.suggest_float("random_strength", 0.0, 2.0),
        bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 1.0),
        loss_function="RMSE", random_seed=42, thread_count=4,
        early_stopping_rounds=100, verbose=0,
    )
    Xtr, Xva = train_x[feat_cb].copy(), val_x[feat_cb].copy()
    for c in cat_cols_cb:
        Xtr[c] = Xtr[c].astype(str); Xva[c] = Xva[c].astype(str)
    m = CatBoostRegressor(**params)
    m.fit(Xtr, np.log1p(train_x[TGT]), cat_features=cat_cols_cb,
          eval_set=(Xva, np.log1p(y_va)), use_best_model=True)
    p = np.clip(np.expm1(m.predict(Xva)) * 0.88, 100000, 1000000)  # с нашим k
    return mape_safe(y_va, p)

study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f"\nЛучший MAPE на val (k=0.88): {study.best_value:.2f}%")
print(f"Дефолт был:                  {base_mape:.2f}%")
print(f"Лучшие параметры: {study.best_params}")

Дефолтный CatBoost + k=0.88: 15.01%



Best trial: 6. Best value: 15.0033: 100%|██████████| 20/20 [08:11<00:00, 24.55s/it]


Лучший MAPE на val (k=0.88): 15.00%
Дефолт был:                  15.01%
Лучшие параметры: {'learning_rate': 0.04050837781329675, 'depth': 4, 'l2_leaf_reg': 1.5854643368675156, 'random_strength': 1.8977710745066665, 'bagging_temperature': 0.9656320330745594}


In [ ]:
# проверка переноса на old→young с лучшими параметрами:

# Обучаем CatBoost с лучшими параметрами на old, проверяем young
best_params = dict(iterations=3000, loss_function="RMSE", random_seed=42,
                   thread_count=4, early_stopping_rounds=100, verbose=0,
                   **study.best_params)
o = train_x[train_x["MONTH_COUNT"] >= 60]
yo = train_x[train_x["MONTH_COUNT"] < 60]

Xo, Xy = o[feat_cb].copy(), yo[feat_cb].copy()
for c in cat_cols_cb:
    Xo[c] = Xo[c].astype(str); Xy[c] = Xy[c].astype(str)
m_best_oy = CatBoostRegressor(**best_params)
m_best_oy.fit(Xo, np.log1p(o[TGT]), cat_features=cat_cols_cb,
              eval_set=(Xy, np.log1p(yo[TGT])), use_best_model=True)
p_oy = np.clip(np.expm1(m_best_oy.predict(Xy)) * 0.88, 100000, 1000000)

print("old→young дефолт:        15.60%")
print(f"old→young оттюненный:    {mape_safe(yo[TGT].to_numpy(), p_oy):.2f}%")

old→young дефолт:        15.60%
old→young оттюненный:    15.49%


Берём оттюненные параметры. Обоснование:

- val: 15.00 (не хуже дефолта),
- old→young: 15.49 (лучше на 0.11) — переносится,
- более простая модель (depth=4) надёжнее под дрейф.
 
1. Модель:   `CatBoost` оттюненный:
          lr=0.0405, depth=4, l2_leaf_reg=1.585,
          random_strength=1.898, bagging_temperature=0.966,
          iterations=3000, loss=RMSE, seed=42, thread_count=4

2. Фичи:     feat_cb (52, сырой CITY, без MONTH_COUNT)

3. Таргет:   TARGET_2 (основной) + TARGET_1 (страховка)

4. Пост:     clip(expm1(pred) × 0.88, 100_000, 1_000_000)

5. val MAPE: 15.00%   old→young: 15.49%

#### Final

In [33]:
# ============ ФИНАЛ: обучение на полном train + прогноз на holdout ============
# Параметры зафиксированы явно (НЕ через study) — для воспроизводимости в Docker

BEST_PARAMS = dict(
    iterations=3000,
    learning_rate=0.04050837781329675,
    depth=4,
    l2_leaf_reg=1.5854643368675156,
    random_strength=1.8977710745066665,
    bagging_temperature=0.9656320330745594,
    loss_function="RMSE",
    random_seed=42,
    thread_count=4,
    early_stopping_rounds=100,
    verbose=0,
)
K_MULTIPLIER = 0.88                         # коррекция на concept drift (подобрана на val)
CLIP_LOW, CLIP_HIGH = 100_000, 1_000_000

def fit_predict_final(target_col, train_df, holdout_df, val_df):
    """
    Обучает CatBoost на полном train для target_col, прогнозирует holdout.
    Использует val как eval_set для early stopping (НЕ для подбора — параметры фиксированы).
    Возвращает прогноз на holdout (с k и clip).
    """
    Xtr = train_df[feat_cb].copy()
    Xva = val_df[feat_cb].copy()
    Xho = holdout_df[feat_cb].copy()
    for c in cat_cols_cb:
        Xtr[c] = Xtr[c].astype(str)
        Xva[c] = Xva[c].astype(str)
        Xho[c] = Xho[c].astype(str)

    m = CatBoostRegressor(**BEST_PARAMS)
    m.fit(Xtr, np.log1p(train_df[target_col]), cat_features=cat_cols_cb,
          eval_set=(Xva, np.log1p(val_df[target_col])), use_best_model=True)

    # прогноз на holdout: expm1 → ×k → clip
    pred = np.clip(np.expm1(m.predict(Xho)) * K_MULTIPLIER, CLIP_LOW, CLIP_HIGH)
    # для контроля — прогноз на val (с теми же k, clip)
    pred_val = np.clip(np.expm1(m.predict(Xva)) * K_MULTIPLIER, CLIP_LOW, CLIP_HIGH)
    val_mape = mape_safe(val_df[target_col].to_numpy(), pred_val)
    return m, pred, val_mape

# Обучаем обе модели (T2 — основная, T1 — страховка)
m_t2, pred_t2, vmape_t2 = fit_predict_final("TARGET_2", train_x, holdout_x, val_x)
m_t1, pred_t1, vmape_t1 = fit_predict_final("TARGET_1", train_x, holdout_x, val_x)

print(f"Финальный val MAPE (с k={K_MULTIPLIER}):")
print(f"  TARGET_2: {vmape_t2:.2f}%   ← основная")
print(f"  TARGET_1: {vmape_t1:.2f}%   ← страховка")
print(f"\nholdout прогнозов: {len(pred_t2)} (ожидаем 1322)")
print(f"Диапазон T2: [{pred_t2.min():.0f}, {pred_t2.max():.0f}]")

Финальный val MAPE (с k=0.88):
  TARGET_2: 15.00%   ← основная
  TARGET_1: 14.96%   ← страховка

holdout прогнозов: 1322 (ожидаем 1322)
Диапазон T2: [135428, 479767]


Диапазон прогноза — здоровый

- [135428, 479767] — полностью внутри [100k, 1M], не упирается в clip-границы (значит модель не выдаёт экстремумов, clip почти не срабатывает). Медиана holdout-прогноза должна быть около 240k (как train) — похоже на правду. Хороший признак.

Sanity — всё корректно

- 1322 прогноза = размер holdout ✓
- диапазон валидный, без отрицательных/нулей ✓
- k применён (иначе диапазон был бы выше) ✓

In [34]:
print(holdout_x.columns.tolist())

['ID', 'REGION', 'BRANCH', 'CITY', 'SUBJECT', 'SUBFRMT', 'TRADE_SQUARE', 'HUFFFAMILIES', 'HUFFRELATIVEFAMILIES', 'HOSPITAL', 'KINDERGARTENS', 'BANKS', 'MK_ON_MM', 'ROUTES_CNT', 'CROSSWALK', 'CROSSROAD', 'DENSITY_FAMILY_TYPE', 'COLLEGES', 'HUFF_HOTEL', 'TRADING_PATCH_SCORE', 'RENT_HEX', 'IN_COAL_CITY', 'IN_OIL_CITY', 'CITY_TYPE', 'ON_DUPLICATE_ROAD', 'HUFF_RANK_COMPS_CANNIBALS', 'TOTAL_RANK_COMPS_CANNIBALS', 'INSIDE_YARD', 'ON_INSIDE_DISTRICT_ROAD', 'ON_MAIN_CITY_ROAD', 'ON_INTERCITY_HIGHWAY', 'ENTRANCE_TO_DISTRICT', 'ON_DISTRICT_BOTTOM', 'HUB', 'ALCOHOL', 'TOBACCO', 'LOCATION_MARKET', 'STATIONS', 'TRC', 'METRO', 'LOCATION_PARK', 'MINI_TRC', 'TRADING_PATCH', 'ENTIRE_DAY', 'MORNING_ROADSIDE', 'SEA', 'FLOORS_TZ', 'SNT', 'LOCATION_TYPE', 'DIST_TO_ADM_CENTER', 'PARKING', 'MONTH_COUNT', 'NEW_BUILDINGS', 'CHANGE_CA', 'CITY_FREQ', 'FAMILIES_PER_SQ', 'RELATIVE_FAM_SQ', 'RENT_DENSITY', 'COMP_PRESSURE', 'INFRA_SCORE', 'TRANSPORT_ACCESS', 'HAS_LICENSE', 'SQUARE_LOG', 'DIST_TO_CENTER_LOG', 'RENT_RE

In [35]:
import os
print(os.getcwd())

/Users/alexey_macos/Documents/ITjob/Pet-projects/Magnit_retailhack_2026/notebooks


In [36]:
OUT_DIR = Path().resolve().parent 
print(f"Результаты сохраняются в: {OUT_DIR}")

Результаты сохраняются в: /Users/alexey_macos/Documents/ITjob/Pet-projects/Magnit_retailhack_2026


In [ ]:
ID_COL = "ID"

# Сборка
sub_t2 = pd.DataFrame({ID_COL: holdout_x[ID_COL].to_numpy(), "PREDICT": pred_t2})  # TARGET_2 — основной
sub_t1 = pd.DataFrame({ID_COL: holdout_x[ID_COL].to_numpy(), "PREDICT": pred_t1})  # TARGET_1 — страховка

# --- Контроль качества ---
for name, sub in [("T2", sub_t2), ("T1", sub_t1)]:
    assert len(sub) == len(holdout_x), f"{name}: длина не совпадает с holdout!"
    assert sub["PREDICT"].isna().sum() == 0, f"{name}: есть NaN!"
    assert (sub["PREDICT"] < 0).sum() == 0, f"{name}: есть отрицательные!"
    assert sub[ID_COL].equals(holdout_x[ID_COL]), f"{name}: порядок ID нарушен!"
print("Все проверки формата пройдены ✓")

# --- Сохранение в корень проекта (UTF-8, запятая, заголовок) ---
sub_t2.to_csv(OUT_DIR / "predictions.csv", index=False, encoding="utf-8")          # ← официальный submit (ТЗ)
sub_t2.to_csv(OUT_DIR / "predictions_target2.csv", index=False, encoding="utf-8")  # понятная копия
sub_t1.to_csv(OUT_DIR / "predictions_target1.csv", index=False, encoding="utf-8")  # страховка T1

print("\npredictions.csv (= TARGET_2) — основной submit:")
print(sub_t2.head())
print(f"\nСохранено в {OUT_DIR}:")
print(f"  predictions.csv         — СДАЁМ это (= TARGET_2, {len(sub_t2)} строк)")
print("  predictions_target2.csv — копия с понятным именем")
print(f"  predictions_target1.csv — страховка TARGET_1 ({len(sub_t1)} строк)")

Все проверки формата пройдены ✓

predictions.csv (= TARGET_2) — основной submit:
      ID        PREDICT
0  17178  191516.348236
1  17179  270069.245236
2  17180  216116.399163
3  17181  330103.562170
4  17182  192069.364233

Сохранено в /Users/alexey_macos/Documents/ITjob/Pet-projects/Magnit_retailhack_2026:
  predictions.csv         — СДАЁМ это (= TARGET_2, 1322 строк)
  predictions_target2.csv — копия с понятным именем
  predictions_target1.csv — страховка TARGET_1 (1322 строк)


In [37]:
# Недоторговка считается на VAL (где есть факт). На holdout факта нет.
# Сравниваем k=1.0 vs k=0.88 — показать, что коррекция улучшает бизнес-метрику.
SEGMENT_THRESHOLD = 246083

Xva = val_x[feat_cb].copy()
for c in cat_cols_cb:
    Xva[c] = Xva[c].astype(str)
pred_val_raw = np.clip(np.expm1(m_t2.predict(Xva)), CLIP_LOW, CLIP_HIGH)  # k=1
pred_val_k   = np.clip(pred_val_raw * K_MULTIPLIER, CLIP_LOW, CLIP_HIGH)  # k=0.88
y_va = val_x["TARGET_2"].to_numpy()

def undertrade(y_true, y_pred):
    """Доля магазинов, где факт < 90% прогноза (недоторговка по ТЗ)."""
    return (y_true < 0.9 * y_pred).mean() * 100

print("Недоторговка (факт < 90% прогноза), TARGET_2 на val:")
print(f"  без коррекции (k=1.00): {undertrade(y_va, pred_val_raw):.1f}%")
print(f"  с коррекцией  (k=0.88): {undertrade(y_va, pred_val_k):.1f}%")

print("\nПо ценовым сегментам (k=0.88):")
for name, mask in [("low  <246083", y_va < SEGMENT_THRESHOLD),
                   ("high ≥246083", y_va >= SEGMENT_THRESHOLD)]:
    print(f"  {name}: недоторговка={undertrade(y_va[mask], pred_val_k[mask]):.1f}%, "
          f"MAPE={mape_safe(y_va[mask], pred_val_k[mask]):.2f}% (n={mask.sum()})")

Недоторговка (факт < 90% прогноза), TARGET_2 на val:
  без коррекции (k=1.00): 47.4%
  с коррекцией  (k=0.88): 22.9%

По ценовым сегментам (k=0.88):
  low  <246083: недоторговка=31.9%, MAPE=13.52% (n=1381)
  high ≥246083: недоторговка=4.8%, MAPE=17.99% (n=687)


#### Гипотеза: Заменить статический k=0.88 на динамический k(MONTH_COUNT).

In [39]:
# Динамический множитель: чем моложе магазин (меньше MONTH_COUNT), тем сильнее занижаем.
# MONTH_COUNT используем ТОЛЬКО для пост-коррекции, НЕ как фичу модели (там он leak).

def dynamic_k(month_count, k_young=0.85, k_old=0.95, m_min=1, m_max=36):
    """
    Линейная интерполяция k по возрасту:
    - очень молодой магазин (m_min) → k_young (сильное занижение)
    - старый по меркам holdout (m_max) → k_old (слабое занижение)
    Клип за границами диапазона.
    """
    m = np.clip(month_count, m_min, m_max)
    frac = (m - m_min) / (m_max - m_min)
    return k_young + (k_old - k_young) * frac

# Подбор параметров k_young/k_old на val (val MONTH_COUNT 13-36)
y_va = val_x["TARGET_2"].to_numpy()
Xva = val_x[feat_cb].copy()
for c in cat_cols_cb:
    Xva[c] = Xva[c].astype(str)
pred_val_base = np.clip(np.expm1(m_t2.predict(Xva)), CLIP_LOW, CLIP_HIGH)  # без k

print("Подбор динамического k на val:\n")

best = (None, 1e9)
for ky in np.arange(0.82, 0.95, 0.02):
    for ko in np.arange(ky, 1.01, 0.02):
        kv = dynamic_k(val_x["MONTH_COUNT"].to_numpy(), k_young=ky, k_old=ko)
        p = np.clip(pred_val_base * kv, CLIP_LOW, CLIP_HIGH)
        mp = mape_safe(y_va, p)
        if mp < best[1]:
            best = ((round(ky,2), round(ko,2)), mp)

print(f"Лучшие (k_young, k_old)={best[0]}: val MAPE={best[1]:.2f}%")
print("Статический k=0.88: 15.00% (референс)")

# Сравнение со статическим на old→young (проверка стабильности)
o = train_x[train_x["MONTH_COUNT"] >= 60]
yo = train_x[train_x["MONTH_COUNT"] < 60]
Xo, Xy = o[feat_cb].copy(), yo[feat_cb].copy()
for c in cat_cols_cb:
    Xo[c] = Xo[c].astype(str); Xy[c] = Xy[c].astype(str)
m_oy = CatBoostRegressor(**BEST_PARAMS)
m_oy.fit(Xo, np.log1p(o[TGT]), cat_features=cat_cols_cb,
         eval_set=(Xy, np.log1p(yo[TGT])), use_best_model=True)
p_oy_base = np.clip(np.expm1(m_oy.predict(Xy)), CLIP_LOW, CLIP_HIGH)
ky, ko = best[0]
kv_oy = dynamic_k(yo["MONTH_COUNT"].to_numpy(), k_young=ky, k_old=ko)

print(f"\nold→young динамический k: {mape_safe(yo[TGT].to_numpy(), np.clip(p_oy_base*kv_oy, CLIP_LOW, CLIP_HIGH)):.2f}%")
print("old→young статический 0.88: 15.49% (референс)")

Подбор динамического k на val:

Лучшие (k_young, k_old)=(np.float64(0.82), np.float64(0.92)): val MAPE=14.73%
Статический k=0.88: 15.00% (референс)

old→young динамический k: 14.70%
old→young статический 0.88: 15.49% (референс)


!!!

Динамический k улучшает обе метрики, причём на old→young сильнее (−0.79).

Проверка динамического k с границами:  для m_min=1, m_max=36

In [40]:
y_va = val_x["TARGET_2"].to_numpy()
mc_va = val_x["MONTH_COUNT"].to_numpy()

# Подбор k_young/k_old при РАЗНЫХ границах
print("Границы (m_min,m_max)   лучшие (k_young,k_old)   val MAPE")
configs = [(1, 36), (1, 24), (13, 36), (1, 12)]
results = {}
for m_min, m_max in configs:
    best = (None, 1e9)
    for ky in np.arange(0.80, 0.95, 0.01):
        for ko in np.arange(ky, 1.01, 0.01):
            kv = dynamic_k(mc_va, k_young=ky, k_old=ko, m_min=m_min, m_max=m_max)
            mp = mape_safe(y_va, np.clip(pred_val_base * kv, CLIP_LOW, CLIP_HIGH))
            if mp < best[1]:
                best = ((round(ky,2), round(ko,2)), mp)
    results[(m_min, m_max)] = best
    print(f"  ({m_min:2d},{m_max:2d})              {best[0]}        {best[1]:.2f}%")

# old→young проверка для лучшей конфигурации
print("\nold→young проверка (стабильность):")
mc_yo = yo["MONTH_COUNT"].to_numpy()
for (m_min, m_max), (params, _) in results.items():
    ky, ko = params
    kv = dynamic_k(mc_yo, k_young=ky, k_old=ko, m_min=m_min, m_max=m_max)
    oy = mape_safe(yo[TGT].to_numpy(), np.clip(p_oy_base * kv, CLIP_LOW, CLIP_HIGH))
    print(f"  ({m_min:2d},{m_max:2d}) {params}: old→young={oy:.2f}%")

Границы (m_min,m_max)   лучшие (k_young,k_old)   val MAPE
  ( 1,36)              (np.float64(0.8), np.float64(0.93))        14.69%
  ( 1,24)              (np.float64(0.8), np.float64(0.9))        14.78%
  (13,36)              (np.float64(0.83), np.float64(0.95))        14.66%
  ( 1,12)              (np.float64(0.8), np.float64(0.88))        15.00%

old→young проверка (стабильность):
  ( 1,36) (np.float64(0.8), np.float64(0.93)): old→young=14.61%
  ( 1,24) (np.float64(0.8), np.float64(0.9)): old→young=15.01%
  (13,36) (np.float64(0.83), np.float64(0.95)): old→young=14.58%
  ( 1,12) (np.float64(0.8), np.float64(0.88)): old→young=15.49%


In [41]:
# ============ ФИНАЛЬНАЯ КОНФИГУРАЦИЯ ============
BEST_PARAMS = dict(
    iterations=3000, 
    learning_rate=0.04050837781329675, 
    depth=4,
    l2_leaf_reg=1.5854643368675156, 
    random_strength=1.8977710745066665,
    bagging_temperature=0.9656320330745594, 
    loss_function="RMSE",
    random_seed=42, 
    thread_count=4, 
    early_stopping_rounds=100, 
    verbose=0,
)

CLIP_LOW, CLIP_HIGH = 100_000, 1_000_000
K_PARAMS = dict(k_young=0.80, k_old=0.93, m_min=1, m_max=36)

def dynamic_k(month_count, k_young, k_old, m_min, m_max):
    """Чем моложе магазин — тем сильнее занижаем (старички завышают прогноз новым).
    MONTH_COUNT здесь ТОЛЬКО для пост-калибровки, НЕ фича модели (там он leak)."""
    m = np.clip(month_count, m_min, m_max)
    frac = (m - m_min) / (m_max - m_min)
    return k_young + (k_old - k_young) * frac

def fit_predict_dynamic(target_col, train_df, holdout_df, val_df):
    """Обучает CatBoost, прогнозирует holdout с динамической калибровкой по возрасту."""
    Xtr, Xva, Xho = (df[feat_cb].copy() for df in (train_df, val_df, holdout_df))
    for c in cat_cols_cb:
        Xtr[c], Xva[c], Xho[c] = Xtr[c].astype(str), Xva[c].astype(str), Xho[c].astype(str)

    m = CatBoostRegressor(**BEST_PARAMS)
    m.fit(Xtr, np.log1p(train_df[target_col]), cat_features=cat_cols_cb,
          eval_set=(Xva, np.log1p(val_df[target_col])), use_best_model=True)

    # динамический k по MONTH_COUNT каждого магазина
    k_ho = dynamic_k(holdout_df["MONTH_COUNT"].to_numpy(), **K_PARAMS)
    k_va = dynamic_k(val_df["MONTH_COUNT"].to_numpy(), **K_PARAMS)
    pred_ho = np.clip(np.expm1(m.predict(Xho)) * k_ho, CLIP_LOW, CLIP_HIGH)
    pred_va = np.clip(np.expm1(m.predict(Xva)) * k_va, CLIP_LOW, CLIP_HIGH)
    return m, pred_ho, mape_safe(val_df[target_col].to_numpy(), pred_va)

# Обучаем обе модели
m_t2, pred_t2, vmape_t2 = fit_predict_dynamic("TARGET_2", train_x, holdout_x, val_x)
m_t1, pred_t1, vmape_t1 = fit_predict_dynamic("TARGET_1", train_x, holdout_x, val_x)

print(f"Финальный val MAPE (динамический k {K_PARAMS}):\n")
print(f"  TARGET_2: {vmape_t2:.2f}%  ← основная")
print(f"  TARGET_1: {vmape_t1:.2f}%  ← страховка")

Финальный val MAPE (динамический k {'k_young': 0.8, 'k_old': 0.93, 'm_min': 1, 'm_max': 36}):

  TARGET_2: 14.69%  ← основная
  TARGET_1: 14.76%  ← страховка


In [ ]:
# проверка формата predictions

ID_COL = "ID"

sub_t2 = pd.DataFrame({ID_COL: holdout_x[ID_COL].to_numpy(), "PREDICT": pred_t2})
sub_t1 = pd.DataFrame({ID_COL: holdout_x[ID_COL].to_numpy(), "PREDICT": pred_t1})

for name, sub in [("T2", sub_t2), ("T1", sub_t1)]:
    assert len(sub) == len(holdout_x), f"{name}: длина!"
    assert sub["PREDICT"].isna().sum() == 0, f"{name}: NaN!"
    assert (sub["PREDICT"] < 0).sum() == 0, f"{name}: отрицательные!"
    assert sub[ID_COL].equals(holdout_x[ID_COL]), f"{name}: порядок ID!"
print("Все проверки формата пройдены ✓")

sub_t2.to_csv(OUT_DIR / "predictions.csv", index=False, encoding="utf-8")
sub_t2.to_csv(OUT_DIR / "predictions_target2.csv", index=False, encoding="utf-8")
sub_t1.to_csv(OUT_DIR / "predictions_target1.csv", index=False, encoding="utf-8")

print(f"\nДиапазон T2: [{pred_t2.min():.0f}, {pred_t2.max():.0f}]")
print(f"Сохранено в {OUT_DIR}: predictions.csv (= TARGET_2, {len(sub_t2)} строк)")
print(sub_t2.head())

Все проверки формата пройдены ✓

Диапазон T2: [124831, 442226]
Сохранено в /Users/alexey_macos/Documents/ITjob/Pet-projects/Magnit_retailhack_2026: predictions.csv (= TARGET_2, 1322 строк)
      ID        PREDICT
0  17178  182997.601577
1  17179  258056.424912
2  17180  206503.429460
3  17181  315420.384243
4  17182  183526.019136
